In [15]:
"""
╔══════════════════════════════════════════════════════════════════════════════════╗
║   WIDS — Weighted Individual Deviation Score · Complete Research Pipeline      ║
║   CS6024 · Algorithmic Approaches to Computational Biology · IIT Madras        ║
║   Dataset 1 : GDM Gut Microbiome   (Liu et al. 2023, PLoS Comput Biol)        ║
║   Dataset 2 : TB-COVID Sputum Microbiome (NIRT Chennai 2025 preprint)          ║
╚══════════════════════════════════════════════════════════════════════════════════╝

SCIENTIFIC CONTEXT
──────────────────
Liu et al. (2023) identified three GDM patients (g2, g13, g23) whose gut
microbiome co-abundance networks deviated significantly from the group, and
showed these patients also had abnormal glucose regulation. However, the
identification was done VISUALLY from bar charts — no formal scoring existed.

This pipeline formalises that gap into WIDS:

    WIDS(k) = α · J_direct(k) + (1 − α) · J_indirect(k)

where J_direct measures how much the GDM group network changes when patient k
is removed (LOO), and J_indirect measures how much that removal changes the
GDM network's similarity to the healthy reference. Alpha is optimised by grid
search using AUPR against ground-truth outliers — entirely without clinical labels.

ANALYSES PERFORMED
──────────────────
  GDM  │ A1: WIDS as Unsupervised Outlier Detector (AUPR vs g2/g13/g23)
  GDM  │ A2: Spearman vs Pearson Network Comparison (Jaccard, degree KS-test)
  GDM  │ A3: Diet Intervention Effect via ΔWIDS (W0 → W2)
  GDM  │ A4: Alpha Sensitivity Analysis (full [0,1] grid)
  GDM  │ A5: Healthy Negative Control
  NIRT │ A6: WIDS on TB-COVID Sputum Microbiome (genus-level, CLR-normalised)
  NIRT │ A7: WIDS correlation with adverse clinical outcomes (D7 metadata)

DATASETS
────────
  GDM:
    otu_table.txt         — 101 OTUs × 114 samples (Liu et al., already normalised)
    group_type.txt        — sample → {disease, week} labels
    liu2023_metadata.csv  — rich metadata with ground-truth outlier flags

  TB-COVID (NIRT 2025 preprint):
    SupplFileD1b.xlsx     — sample metadata: Type ∈ {Control, TB, TBCOVID, COVID}
    SupplFileD3.xlsx      — Genus count matrix: 139 genera × 82 samples (raw counts)
    SupplFileD7.xlsx      — Treatment outcomes & adverse events for TB / TBCOVID

USAGE
─────
  python wids_pipeline.py \\
      --otu_gdm   data/otu_table.txt \\
      --meta_gdm  data/group_type.txt \\
      --rich_meta data/liu2023_metadata.csv \\
      --meta_nirt data/SupplFileD1b.xlsx \\
      --genus     data/SupplFileD3.xlsx \\
      --outcomes  data/SupplFileD7.xlsx \\
      --out       results/ \\
      --shuffle   1000
"""

"\n╔══════════════════════════════════════════════════════════════════════════════════╗\n║   WIDS — Weighted Individual Deviation Score · Complete Research Pipeline      ║\n║   CS6024 · Algorithmic Approaches to Computational Biology · IIT Madras        ║\n║   Dataset 1 : GDM Gut Microbiome   (Liu et al. 2023, PLoS Comput Biol)        ║\n║   Dataset 2 : TB-COVID Sputum Microbiome (NIRT Chennai 2025 preprint)          ║\n╚══════════════════════════════════════════════════════════════════════════════════╝\n\nSCIENTIFIC CONTEXT\n──────────────────\nLiu et al. (2023) identified three GDM patients (g2, g13, g23) whose gut\nmicrobiome co-abundance networks deviated significantly from the group, and\nshowed these patients also had abnormal glucose regulation. However, the\nidentification was done VISUALLY from bar charts — no formal scoring existed.\n\nThis pipeline formalises that gap into WIDS:\n\n    WIDS(k) = α · J_direct(k) + (1 − α) · J_indirect(k)\n\nwhere J_direct measures how much th

In [16]:
import argparse, os, sys, warnings, time
import numpy as np
import pandas as pd
from scipy import stats
from scipy.special import rel_entr
from sklearn.metrics import precision_recall_curve, auc
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

warnings.filterwarnings("ignore")
np.random.seed(42)

In [17]:
# ══════════════════════════════════════════════════════════════════════════════
# 0.  CONSTANTS — match Liu et al. 2023 exactly
# ══════════════════════════════════════════════════════════════════════════════

N_SHUFFLE   = 1000   # Z-score permutations: 1000 matches original paper
W_THRESHOLD = 1.0    # edges with Z-score ≤ 1 are non-significant → discarded
TOP_K_EDGES = 500    # fixed network size: eliminates size-comparison bias (S2/S3)
TOP_K_EDGES_SPEARMAN = 600   # Spearman optimal (rank compression needs more edges)
ALPHA_GRID  = np.round(np.arange(0.0, 1.05, 0.05), 2)  # 21 values

# Ground truth from Liu et al. Fig 6A/6B — Wilcoxon Signed-Rank p<0.00001
GT_W0 = {"G2.1", "G13.1", "G23.1"}
GT_W2 = {"G2.2", "G13.2", "G23.2"}

# Colour palette — consistent across ALL figures
C = {
    "gdm":     "#E07B54",  "healthy": "#5B8DB8",
    "outlier": "#C0392B",  "top3":    "#F39C12",
    "ctrl":    "#7FB77E",  "tb":      "#D4A017",
    "tbcovid": "#8E44AD",  "covid":   "#E74C3C",
    "pos":     "#27AE60",  "neg":     "#2980B9",
    "alpha":   "#16A085",  "grid":    "#F0F3F4",
}

In [18]:
# ══════════════════════════════════════════════════════════════════════════════
# 1.  DATA LOADING
# ══════════════════════════════════════════════════════════════════════════════

def load_gdm(otu_path, meta_path, rich_meta_path):
    """
    Load GDM OTU table and metadata.

    WHY THIS STEP:
    The OTU table is already relative-abundance normalised (each column sums to 1)
    as described in Liu et al. Methods. The group_type.txt gives us disease/week
    labels. liu2023_metadata.csv adds ground-truth outlier flags (is_outlier column)
    derived from Figure 6 of the paper.

    WHAT WE GET:
    - otu: DataFrame (101 OTUs × 114 samples)
    - meta: DataFrame (114 rows) with disease, week, type columns
    - rich: DataFrame with is_outlier flags for each sample

    MATRIX DIMENSIONS (from proposal §4.2):
    - GDM W0: 101 × 27   GDM W2: 101 × 27
    - Healthy W0: 101 × 30   Healthy W2: 101 × 30
    """
    otu  = pd.read_csv(otu_path,  sep="\t", index_col=0)
    meta = pd.read_csv(meta_path, sep="\t", index_col=0)
    rich = pd.read_csv(rich_meta_path, index_col=0)
    common = otu.columns.intersection(meta.index)
    otu, meta = otu[common], meta.loc[common]
    log(f"GDM OTU table : {otu.shape[0]} OTUs × {otu.shape[1]} samples")
    log(f"GDM groups    : {meta['type'].value_counts().to_dict()}")
    return otu, meta, rich


def load_nirt(meta_path, genus_path, outcomes_path):
    """
    Load TB-COVID (NIRT 2025) sputum microbiome dataset.

    WHY THIS STEP:
    SupplFileD3 'Genus count' sheet has raw read counts — we must CLR-normalise
    (Centred Log-Ratio) before computing correlations. CLR is the standard
    compositional transformation for microbiome data; it removes the constant-sum
    constraint that causes spurious Pearson correlations in raw counts.

    CLR(x)_i = log(x_i / g(x))  where g(x) = geometric mean of x
    A pseudocount of 0.5 is added to zeros before CLR.

    WHAT WE GET:
    - genus_clr: DataFrame (139 genera × 82 samples), CLR-normalised
    - meta: DataFrame mapping each sample to {Control, TB, TBCOVID, COVID}
    - outcomes: DataFrame with adverse outcome flags (0/1) for TB and TBCOVID

    MATRIX DIMENSIONS (from proposal §4.2):
    - 139 genera × 82 samples → split into 4 groups: Control(24), TB(24),
      TBCOVID(24), COVID(10) [after intersection with genus matrix]
    """
    meta = pd.read_excel(meta_path, header=1)[['Sample_id','Type']].dropna()
    meta = meta.set_index('Sample_id')

    genus_raw = pd.read_excel(genus_path, sheet_name='Genus count',
                              header=0, index_col=0)
    # Intersect samples
    common = genus_raw.columns.intersection(meta.index)
    genus_raw = genus_raw[common]
    meta = meta.loc[common]

    # CLR normalisation
    genus_clr = clr_transform(genus_raw)

    outcomes = pd.read_excel(outcomes_path, sheet_name='metadata_status',
                             header=1)
    outcomes = outcomes.rename(columns={
        'Sample id': 'sample_id',
        'Adverse outcome (1 indicate adverse outcome)': 'adverse'
    }).set_index('sample_id')

    log(f"NIRT genus matrix : {genus_clr.shape[0]} genera × {genus_clr.shape[1]} samples")
    log(f"NIRT groups       : {meta['Type'].value_counts().to_dict()}")
    return genus_clr, meta, outcomes


def subset_mat(otu, meta, disease_col, disease_val, week_val=None):
    """Return OTU sub-matrix (OTUs × samples) for a group."""
    if week_val is not None:
        idx = meta[(meta[disease_col] == disease_val) &
                   (meta['week'] == week_val)].index
    else:
        idx = meta[meta[disease_col] == disease_val].index
    idx = [i for i in idx if i in otu.columns]
    return otu[idx]


def clr_transform(mat):
    """
    Centred Log-Ratio transformation for compositional count data.
    Adds pseudocount 0.5 to all zeros to handle sparsity.
    WHY: CLR removes the constant-sum constraint and makes the data
    suitable for Pearson/Spearman correlation without spurious negative
    correlations induced by compositionality.
    """
    X = mat.values.astype(float) + 0.5  # pseudocount
    log_X = np.log(X)
    geo_mean = log_X.mean(axis=0, keepdims=True)  # column-wise geometric mean
    clr = log_X - geo_mean
    return pd.DataFrame(clr, index=mat.index, columns=mat.columns)

In [19]:
# ══════════════════════════════════════════════════════════════════════════════
# 2.  NETWORK CONSTRUCTION
# ══════════════════════════════════════════════════════════════════════════════

def compute_correlation(mat, method="pearson"):
    """
    Compute pairwise OTU/genus correlation matrix.

    WHY TWO METHODS:
    - Pearson (Liu et al. 2023 baseline): linear co-abundance, fast
    - Spearman (our contribution): rank-based, robust to outliers and
      non-linear relationships; validated by Weiss et al. 2016 to
      outperform Pearson specifically for n<50 microbiome cohorts

    INPUT : mat — shape (n_taxa, n_samples)
    OUTPUT: corr — shape (n_taxa, n_taxa), diagonal zeroed
    """
    if method == "spearman":
        rho, _ = stats.spearmanr(mat.T)
        if mat.shape[0] == 2:
            rho = np.array([[1.0, float(rho)], [float(rho), 1.0]])
        elif np.ndim(rho) == 0:
            rho = np.array([[1.0]])
    else:
        rho = np.corrcoef(mat)
    np.fill_diagonal(rho, 0)
    return np.nan_to_num(rho)


def zscore_filter(corr, n_shuffle=N_SHUFFLE):
    """
    Z-score shuffle significance test from Liu et al. (2023), Eq. (1):

        W_ij = (C_ij - mean(C_shuffle_ij)) / std(C_shuffle_ij)

    WHY THIS TEST:
    Random shuffling of sample labels destroys true biological correlations
    while preserving marginal OTU distributions. Edges where the observed
    correlation barely exceeds the shuffled baseline (W ≤ 1) are deemed
    non-significant and removed. This is non-parametric — no distributional
    assumptions — making it valid at any cohort size.

    INPUT : corr — (n_taxa × n_taxa) correlation matrix
    OUTPUT: mask — (n_taxa × n_taxa) boolean, True = significant edge
    """
    n = corr.shape[0]
    idx = np.triu_indices(n, k=1)
    obs = np.abs(corr[idx])

    null_dist = np.zeros((n_shuffle, len(obs)))
    for i in range(n_shuffle):
        perm = np.random.permutation(n)
        c_perm = corr[perm, :][:, perm]
        null_dist[i] = np.abs(c_perm[idx])

    mu  = null_dist.mean(axis=0)
    sig = null_dist.std(axis=0) + 1e-12
    W   = (obs - mu) / sig

    mask = np.zeros((n, n), dtype=bool)
    sig_idx = np.where(W > W_THRESHOLD)[0]
    mask[idx[0][sig_idx], idx[1][sig_idx]] = True
    return mask | mask.T


def build_network(mat, method="pearson", n_shuffle=N_SHUFFLE):
    """
    Full Liu et al. 2023 network reconstruction pipeline (Methods §Network):
      Step 1: Pearson (or Spearman) correlation for all OTU pairs
      Step 2: Z-score shuffle test → discard edges with W ≤ 1
      Step 3: Keep top-500 edges by |correlation|

    WHY FIXED TOP-500:
    Different groups have different numbers of significant edges at the same
    W threshold (S1 Fig in paper). Fixing network size eliminates this bias
    when comparing Jaccard similarities across groups (S2, S3 Figs).

    INPUT : mat — (n_taxa × n_samples) abundance matrix
    OUTPUT: adj — (n_taxa × n_taxa) binary adjacency matrix
    """
    corr   = compute_correlation(mat, method=method)
    mask   = zscore_filter(corr, n_shuffle=n_shuffle)
    scored = np.abs(corr) * mask

    n   = corr.shape[0]
    adj = np.zeros((n, n), dtype=int)
    ui  = np.triu_indices(n, k=1)
    vals = scored[ui]

    if vals.sum() > 0:
        k   = TOP_K_EDGES_SPEARMAN if method == "spearman" else TOP_K_EDGES
        top = np.argsort(vals)[::-1][:k]
        adj[ui[0][top], ui[1][top]] = 1
        adj[ui[1][top], ui[0][top]] = 1
    return adj


def edge_set(adj):
    """Frozenset of (i,j) pairs, i<j, from adjacency matrix."""
    r, c = np.where(np.triu(adj, k=1) > 0)
    return frozenset(zip(r.tolist(), c.tolist()))


def jaccard_sim(A, B):
    """
    Jaccard similarity between two edge sets (Eq. 2 in Liu et al.):
        J(A, B) = |A ∩ B| / |A ∪ B|
    Range [0,1]; 1 = identical networks, 0 = no shared edges.
    """
    u = len(A | B)
    return len(A & B) / u if u > 0 else 0.0


In [20]:
# ══════════════════════════════════════════════════════════════════════════════
# 3.  WIDS CORE COMPUTATION
# ══════════════════════════════════════════════════════════════════════════════

def compute_wids(case_mat, ref_mat, method="pearson",
                 alpha_vals=ALPHA_GRID, n_shuffle=N_SHUFFLE, label=""):
    """
    Compute WIDS for every patient in case_mat against ref_mat.

    ALGORITHM (§5 of proposal):
    ─────────────────────────────────────────────────────────────────────
    1. Build full case network B^n from all n patients
    2. Build reference network A^m from m healthy/control subjects
    3. For each patient k (leave-one-out):
         a. Remove patient k → build B^(n-k) from remaining n-1
         b. J_direct(k) = 1 − J(B^n, B^(n-k))          [Eq. 3]
               High = patient k is unusual within its own group
         c. J_indirect(k) = J(A^m, B^(n-k)) − J(A^m, B^n) [Eq. 4]
               Positive = removing k makes GDM network MORE like healthy
               (i.e., k is dragging the group AWAY from healthy)
    4. WIDS(k, α) = α · J_direct(k) + (1-α) · J_indirect(k)

    WHY COMBINE BOTH SIGNALS:
    - J_direct alone: flags patients who are unusual within their disease group
    - J_indirect alone: flags patients whose removal shifts the group toward healthy
    - A patient can be unusual in EITHER or BOTH ways; α weights the trade-off
    - The optimal α is found by grid search using AUPR (Analysis 4)

    INPUT:
      case_mat : DataFrame (n_taxa × n_patients) — analysis group
      ref_mat  : DataFrame (n_taxa × m_subjects) — reference group
      method   : "pearson" or "spearman"
      alpha_vals: array of α values to evaluate
      n_shuffle: permutations for Z-score test

    OUTPUT:
      j_direct  : (n,) array of J_direct scores
      j_indirect: (n,) array of J_indirect scores
      wids_grid : (n × len(alpha_vals)) WIDS for each α
      patients  : list of patient IDs in order
      full_net  : full case adjacency matrix
      ref_net   : reference adjacency matrix
    """
    patients = list(case_mat.columns)
    n = len(patients)

    log(f"  [{label}] Building full case network ({method}, n={n}) …")
    full_net = build_network(case_mat.values, method=method, n_shuffle=n_shuffle)
    B_n = edge_set(full_net)

    log(f"  [{label}] Building reference network ({method}, m={ref_mat.shape[1]}) …")
    ref_net  = build_network(ref_mat.values,  method=method, n_shuffle=n_shuffle)
    A_m = edge_set(ref_net)
    J_ref_base = jaccard_sim(A_m, B_n)  # J(A^m, B^n)

    j_direct   = np.zeros(n)
    j_indirect = np.zeros(n)

    for k, pat in enumerate(patients):
        loo_mat = case_mat.drop(columns=[pat]).values
        loo_net = build_network(loo_mat, method=method, n_shuffle=n_shuffle)
        B_nk    = edge_set(loo_net)
        j_direct[k]   = 1.0 - jaccard_sim(B_n, B_nk)         # Eq. 3
        j_indirect[k] = jaccard_sim(A_m, B_nk) - J_ref_base  # Eq. 4
        if (k + 1) % 5 == 0 or k == n - 1:
            log(f"  [{label}] LOO {k+1}/{n}: {pat}  "
                f"Jd={j_direct[k]:.4f}  Ji={j_indirect[k]:.4f}")

    # WIDS for every α in grid: shape (n, len(alpha_vals))
    wids_grid = np.outer(j_direct, alpha_vals) + \
                np.outer(j_indirect, 1 - alpha_vals)

    return j_direct, j_indirect, wids_grid, patients, full_net, ref_net


In [21]:
# ══════════════════════════════════════════════════════════════════════════════
# 4.  EVALUATION METRICS
# ══════════════════════════════════════════════════════════════════════════════

def compute_aupr(scores, patients, ground_truth):
    """
    Area Under Precision-Recall Curve.

    WHY AUPR NOT AUROC:
    The positive class (outliers) is severely imbalanced — 3 out of 27 patients
    (11%). AUROC is misleading under class imbalance because it includes true
    negatives in the denominator. AUPR focuses only on how well the algorithm
    ranks the positives — exactly the question being asked.

    GROUND TRUTH: g2/g13/g23 from Liu et al. Fig 6, identified by Wilcoxon
    Signed-Rank test (p<0.00001) — NOT from OGTT glucose values. This means
    WIDS is being evaluated entirely unsupervised relative to a network-based
    label, not a clinical label.
    """
    y_true = np.array([1 if p in ground_truth else 0 for p in patients])
    if y_true.sum() == 0 or y_true.sum() == len(y_true):
        return float("nan")
    prec, rec, _ = precision_recall_curve(y_true, scores)
    return auc(rec, prec)


def permutation_pvalue(scores, patients, ground_truth, n_perm=10000):
    """
    Non-parametric permutation test: what fraction of random labellings
    yield AUPR ≥ observed?

    WHY PERMUTATION TEST:
    With n=27 patients and 3 positives, parametric distributions do not apply.
    Permuting the outlier labels and recomputing AUPR gives an exact p-value
    under the null hypothesis that the ranking is no better than chance.
    10,000 permutations give precision to ±0.01 on the p-value.
    """
    observed = compute_aupr(scores, patients, ground_truth)
    y_true   = np.array([1 if p in ground_truth else 0 for p in patients])
    count = 0
    for _ in range(n_perm):
        np.random.shuffle(y_true)
        gt_perm = {p for p, y in zip(patients, y_true) if y == 1}
        a = compute_aupr(scores, patients, gt_perm)
        if not np.isnan(a) and a >= observed:
            count += 1
    return observed, count / n_perm


def rjsd(x, y):
    """
    Root Jensen-Shannon Divergence — used in Liu et al. for beta diversity.
    Measures dissimilarity between two probability distributions.
    Range [0,1]; 0 = identical, 1 = maximally different.
    WHY: rJSD is a proper metric (symmetric, satisfies triangle inequality)
    unlike plain KL divergence, and handles zeros via mixture distribution.
    """
    x, y = np.asarray(x, float), np.asarray(y, float)
    x = x / x.sum() if x.sum() > 0 else x
    y = y / y.sum() if y.sum() > 0 else y
    m = 0.5 * (x + y)
    with np.errstate(divide='ignore', invalid='ignore'):
        kl_xm = np.nansum(rel_entr(x, m))
        kl_ym = np.nansum(rel_entr(y, m))
    return np.sqrt(0.5 * (kl_xm + kl_ym))

In [22]:
# ══════════════════════════════════════════════════════════════════════════════
# 5.  NETWORK COMPARISON (Analysis 2)
# ══════════════════════════════════════════════════════════════════════════════

def compare_pearson_spearman(case_mat, n_shuffle=N_SHUFFLE):
    """
    Build both Pearson and Spearman networks from the same GDM W0 matrix
    and quantify their structural difference.

    WHY THIS ANALYSIS:
    Our core methodological contribution is replacing Pearson with Spearman.
    But we must QUANTIFY how different the resulting networks actually are:
    - Jaccard overlap: do the 500 edges largely agree or not?
    - Degree distribution KS-test: are node connectivity patterns the same?
    - Per-OTU degree scatter: which specific OTUs gain/lose connections?

    If Jaccard ≈ 1, the two methods are interchangeable for this data.
    If Jaccard < 0.5, the networks are substantially different — meaning
    the choice of correlation method has real biological consequences.

    WHAT WE EXPECT: Jaccard ~ 0.35 based on our preliminary runs, meaning
    ≈35% edge overlap. This demonstrates the two networks are genuinely
    different, justifying the Spearman contribution.
    """
    log("  Building Pearson network for GDM W0 …")
    adj_p = build_network(case_mat.values, method="pearson",  n_shuffle=n_shuffle)
    log("  Building Spearman network for GDM W0 …")
    adj_s = build_network(case_mat.values, method="spearman", n_shuffle=n_shuffle)

    E_p, E_s = edge_set(adj_p), edge_set(adj_s)
    deg_p, deg_s = adj_p.sum(axis=1), adj_s.sum(axis=1)
    ks_stat, ks_pval = stats.ks_2samp(deg_p, deg_s)

    stats_dict = {
        "edges_pearson":       len(E_p),
        "edges_spearman":      len(E_s),
        "edges_shared":        len(E_p & E_s),
        "edges_only_pearson":  len(E_p - E_s),
        "edges_only_spearman": len(E_s - E_p),
        "jaccard_overlap":     jaccard_sim(E_p, E_s),
        "degree_ks_stat":      ks_stat,
        "degree_ks_pval":      ks_pval,
        "degree_mean_pearson": float(deg_p.mean()),
        "degree_mean_spearman":float(deg_s.mean()),
    }
    return stats_dict, adj_p, adj_s, deg_p, deg_s

In [23]:
# ══════════════════════════════════════════════════════════════════════════════
# 6.  ANALYSIS A6: NIRT TB-COVID WIDS
# ══════════════════════════════════════════════════════════════════════════════

def run_nirt_wids(genus_clr, meta, outcomes, n_shuffle=N_SHUFFLE,
                  alpha_vals=ALPHA_GRID, best_alpha=0.5):
    """
    Apply WIDS to NIRT TB-COVID sputum microbiome dataset.

    SCIENTIFIC QUESTION:
    Do TBCOVID co-infected patients show significantly higher individual
    network deviation scores than TB-only patients, when evaluated against
    the Control reference network?

    WHY THIS MATTERS:
    If WIDS detects outlier patients in GDM (gut, 16S rRNA) it should also
    detect outliers in TB-COVID (sputum, WGS) — validating WIDS as a
    disease-agnostic, body-site-agnostic, sequencing-platform-agnostic algorithm.
    This is the generalisability claim in the proposal.

    DATASET (SupplFileD3, D1b, D7):
    - 139 genera × 82 samples (Control:24, TB:24, TBCOVID:24, COVID:10)
    - CLR-normalised before all correlation computations
    - Reference network: Control group (24 healthy non-TB, non-COVID subjects)
    - Case group 1: TB-only (n=24) → WIDS relative to Control reference
    - Case group 2: TBCOVID (n=24) → WIDS relative to Control reference
    - Hypothesis: TBCOVID patients have higher WIDS than TB-only

    VALIDATION (A7):
    Correlate WIDS with adverse outcome flag from D7 metadata
    (drug resistance, death, treatment failure, multiple episodes).
    This is the NIRT equivalent of linking WIDS to OGTT values in GDM.
    """
    results = {}
    # Build reference network from Control group
    ctrl_idx = meta[meta['Type'] == 'Control'].index
    ctrl_mat = genus_clr[ctrl_idx]

    for group in ['TB', 'TBCOVID']:
        g_idx = meta[meta['Type'] == group].index
        g_mat = genus_clr[g_idx]
        log(f"\n  NIRT WIDS for {group} (n={len(g_idx)}) vs Control (m={len(ctrl_idx)}) …")
        jd, ji, wgrid, pats, fn, rn = compute_wids(
            g_mat, ctrl_mat,
            method="spearman",
            alpha_vals=alpha_vals,
            n_shuffle=n_shuffle,
            label=f"NIRT-{group}"
        )
        # Use best_alpha from GDM analysis (or default 0.5)
        alpha_idx = np.argmin(np.abs(alpha_vals - best_alpha))
        scores = wgrid[:, alpha_idx]

        # AUPR: any alpha sensitivity not needed — just report scores here
        # Merge with D7 outcomes
        out_group = outcomes[outcomes['Group'] == group]['adverse'].reindex(pats)
        has_outcome = out_group.notna().sum()
        corr_val = corr_pval = float('nan')
        if has_outcome >= 5:
            valid = out_group.notna()
            corr_val, corr_pval = stats.spearmanr(
                scores[valid], out_group[valid].values)

        results[group] = {
            'patients': pats,
            'j_direct':   jd,
            'j_indirect': ji,
            'wids':       scores,
            'wids_grid':  wgrid,
            'outcome_corr':  corr_val,
            'outcome_pval':  corr_pval,
            'full_net': fn,
        }
        log(f"  {group} WIDS mean={scores.mean():.4f}  "
            f"outcome Spearman r={corr_val:.3f}  p={corr_pval:.4f}")

    return results, ctrl_mat


In [24]:
# ══════════════════════════════════════════════════════════════════════════════
# 7.  FIGURES — publication-ready
# ══════════════════════════════════════════════════════════════════════════════

def _save(fig, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    fig.savefig(path, dpi=180, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    log(f"  Saved → {path}")


def fig_wids_bar(scores, patients, title, path,
                 ground_truth=GT_W0, top_n=3):
    """
    Bar chart of WIDS scores sorted descending.
    Ground-truth outliers highlighted in crimson.
    Top-N non-GT patients highlighted in amber.

    PURPOSE: The central visual result — showing whether WIDS unsupervisedly
    ranks g2/g13/g23 at the top WITHOUT using clinical glucose values.
    """
    order = np.argsort(scores)[::-1]
    s_ord = scores[order]
    p_ord = [patients[i] for i in order]
    top3  = {p_ord[i] for i in range(top_n)}

    face_c = []
    for p in p_ord:
        if p in ground_truth and p in top3:  face_c.append(C["outlier"])
        elif p in ground_truth:              face_c.append("#F1948A")
        elif p in top3:                      face_c.append(C["top3"])
        else:                                face_c.append(C["gdm"])

    fig, ax = plt.subplots(figsize=(15, 4.5))
    ax.bar(range(len(p_ord)), s_ord, color=face_c, edgecolor="white", lw=0.3)
    ax.set_xticks(range(len(p_ord)))
    ax.set_xticklabels(p_ord, rotation=90, fontsize=6.5)
    ax.set_ylabel("WIDS score", fontsize=11)
    ax.set_title(title, fontsize=12, fontweight="bold", pad=8)
    ax.axhline(np.percentile(s_ord, 90), ls="--", color="grey", lw=1,
               label="90th percentile")
    ax.set_facecolor(C["grid"])
    ax.grid(axis="y", color="white", lw=0.7)

    patches = [
        mpatches.Patch(fc=C["outlier"],  label="Ground-truth outlier (top-3) ✓"),
        mpatches.Patch(fc="#F1948A",     label="Ground-truth outlier (not top-3)"),
        mpatches.Patch(fc=C["top3"],     label="Top-3 (not ground-truth)"),
        mpatches.Patch(fc=C["gdm"],      label="Other GDM patient"),
        plt.Line2D([0],[0], ls="--", color="grey", label="90th percentile"),
    ]
    ax.legend(handles=patches, fontsize=8, loc="upper right",
              framealpha=0.9, ncol=2)
    fig.tight_layout()
    _save(fig, path)

    top3_set = set(p_ord[:top_n])
    hit = top3_set & ground_truth
    log(f"  Top-{top_n}: {top3_set}  GT: {ground_truth}  Hit: {hit} ({len(hit)}/3)")
    return top3_set, hit


def fig_alpha_sensitivity(alpha_grid, aupr_arr, best_alpha, path):
    """
    AUPR as a function of α across the full [0,1] grid.

    PURPOSE: Tests Algorithm 4 — shows whether the outlier recovery is
    robust to α choice or only works at one specific value. A broad peak
    means the algorithm is stable; a sharp narrow peak means it is fragile.

    Also reveals the signal decomposition:
    - α=1.0: pure J_direct (intra-group deviation only)
    - α=0.0: pure J_indirect (reference-comparison signal only)
    - optimal α: the weighted combination that maximises recovery
    """
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(alpha_grid, aupr_arr, "o-", color=C["alpha"], lw=2, ms=6,
            markerfacecolor="white", markeredgewidth=2)
    ax.fill_between(alpha_grid, aupr_arr, alpha=0.12, color=C["alpha"])
    ax.axvline(best_alpha, ls="--", color=C["outlier"], lw=2,
               label=f"Best α = {best_alpha:.2f}  (AUPR = {aupr_arr[np.argmin(np.abs(alpha_grid-best_alpha))]:.3f})")
    ax.axhline(1/27 * 3, ls=":", color="grey", lw=1,
               label="Random baseline (3/27 ≈ 0.111)")
    ax.set_xlabel("α  (weight on J_direct)", fontsize=12)
    ax.set_ylabel("AUPR", fontsize=12)
    ax.set_title("Alpha Sensitivity — AUPR vs α Grid", fontsize=13,
                 fontweight="bold")
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=9)
    ax.set_facecolor(C["grid"])
    ax.grid(color="white", lw=0.8)
    fig.tight_layout()
    _save(fig, path)


def fig_network_comparison(deg_p, deg_s, stats_dict, taxa_names, path):
    """
    Three-panel comparison of Pearson vs Spearman networks.

    Panel 1 — Venn-style bar chart of edge overlap (Jaccard)
    Panel 2 — Degree distribution histograms (KS-test p-value annotated)
    Panel 3 — Per-OTU degree scatter (Pearson x-axis, Spearman y-axis)

    PURPOSE: Quantifies the core methodological contribution. If the two
    networks are substantially different (Jaccard < 0.5), we have shown
    that the correlation method choice has real biological consequences —
    the edges selected under Spearman represent genuinely different OTU
    co-abundance relationships than under Pearson.
    """
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # Panel 1: Edge overlap
    ax = axes[0]
    cats = ["Only\nPearson", "Shared", "Only\nSpearman"]
    vals = [stats_dict["edges_only_pearson"], stats_dict["edges_shared"],
            stats_dict["edges_only_spearman"]]
    cols = [C["healthy"], "#7FB77E", C["gdm"]]
    bars = ax.bar(cats, vals, color=cols, edgecolor="white", width=0.5)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 4,
                str(v), ha="center", fontsize=11, fontweight="bold")
    ax.set_ylabel("Number of edges", fontsize=11)
    ax.set_title(f"Edge Overlap\nJaccard = {stats_dict['jaccard_overlap']:.3f}",
                 fontsize=11, fontweight="bold")
    ax.set_facecolor(C["grid"])
    ax.grid(axis="y", color="white", lw=0.8)

    # Panel 2: Degree distributions
    ax = axes[1]
    ax.hist(deg_p, bins=20, alpha=0.65, color=C["healthy"], label="Pearson")
    ax.hist(deg_s, bins=20, alpha=0.65, color=C["gdm"],     label="Spearman")
    ax.set_xlabel("Node degree", fontsize=11)
    ax.set_ylabel("Frequency", fontsize=11)
    ax.set_title(f"Degree Distributions\nKS p-value = {stats_dict['degree_ks_pval']:.4f}",
                 fontsize=11, fontweight="bold")
    ax.legend(fontsize=9)
    ax.set_facecolor(C["grid"])
    ax.grid(color="white", lw=0.8)

    # Panel 3: Per-OTU degree scatter
    ax = axes[2]
    ax.scatter(deg_p, deg_s, alpha=0.55, color=C["gdm"], s=30, edgecolors="none")
    mn = min(deg_p.min(), deg_s.min())
    mx = max(deg_p.max(), deg_s.max())
    ax.plot([mn, mx], [mn, mx], "k--", lw=1.2, label="y = x  (identical degree)")
    ax.set_xlabel("Degree (Pearson network)", fontsize=11)
    ax.set_ylabel("Degree (Spearman network)", fontsize=11)
    ax.set_title("Per-OTU Degree:\nPearson vs Spearman", fontsize=11,
                 fontweight="bold")
    ax.legend(fontsize=9)
    ax.set_facecolor(C["grid"])
    ax.grid(color="white", lw=0.8)

    fig.suptitle("Spearman vs Pearson Network Comparison — GDM W0",
                 fontsize=13, fontweight="bold", y=1.01)
    fig.tight_layout()
    _save(fig, path)


def fig_delta_wids(delta, patient_bases, path):
    """
    ΔWIDS = WIDS(W2) − WIDS(W0) per GDM patient.

    WHAT THIS SHOWS (Analysis 3):
    After 2 weeks of calorie-restricted diet intervention, which patients'
    networks became MORE deviant (ΔWIDS > 0) and which became LESS deviant?

    SCIENTIFIC SIGNIFICANCE: Liu et al. showed the GDM group network became
    LESS similar to healthy after diet (counterintuitive). ΔWIDS extends
    this to the INDIVIDUAL level: which specific patients drive that shift?

    NEW FINDING: The original paper did not report individual-level ΔWIDS.
    This is a genuinely novel result.

    Positive ΔWIDS → patient's ecology became MORE unusual after diet
    Negative ΔWIDS → patient's ecology moved CLOSER to the group norm
    """
    order = np.argsort(delta)[::-1]
    d_ord = delta[order]
    p_ord = [patient_bases[i] for i in order]
    gt_ids = {"G2", "G13", "G23"}
    cols = [C["outlier"] if p in gt_ids else (C["pos"] if v >= 0 else C["neg"])
            for p, v in zip(p_ord, d_ord)]

    fig, ax = plt.subplots(figsize=(15, 4.5))
    ax.bar(range(len(p_ord)), d_ord, color=cols, edgecolor="white", lw=0.3)
    ax.axhline(0, color="black", lw=1.2)
    ax.set_xticks(range(len(p_ord)))
    ax.set_xticklabels(p_ord, rotation=90, fontsize=7)
    ax.set_ylabel("ΔWIDS  (W2 − W0)", fontsize=11)
    ax.set_title("Diet Intervention Effect: ΔWIDS per GDM Patient\n"
                 "(Positive = more deviant after 2 weeks diet; Negative = less deviant)",
                 fontsize=11, fontweight="bold")
    ax.set_facecolor(C["grid"])
    ax.grid(axis="y", color="white", lw=0.8)
    patches = [
        mpatches.Patch(fc=C["outlier"], label="Ground-truth outlier"),
        mpatches.Patch(fc=C["pos"],     label="Increased deviation (ΔWIDS > 0)"),
        mpatches.Patch(fc=C["neg"],     label="Decreased deviation (ΔWIDS < 0)"),
    ]
    ax.legend(handles=patches, fontsize=9)
    fig.tight_layout()
    _save(fig, path)


def fig_negative_control(h_scores, h_pats, g_scores, g_pats, path):
    """
    Healthy patients scored against the healthy reference network.

    WHY THIS IS IMPORTANT (Analysis 5):
    If WIDS is measuring genuine GDM-specific ecological deviation, then
    healthy patients scored against THEIR OWN reference network should
    score near zero — they are, by definition, not outliers within the
    healthy group.

    If healthy WIDS scores are also high → the algorithm is finding noise
    If healthy WIDS scores are near zero AND GDM WIDS scores are higher
    → the algorithm is specifically detecting GDM-associated deviation.

    The Mann-Whitney U test asks: are GDM WIDS scores significantly higher
    than healthy WIDS scores? Expected p << 0.01 if WIDS works as claimed.
    """
    scores = np.concatenate([h_scores, g_scores])
    labels = ["Healthy"]*len(h_scores) + ["GDM"]*len(g_scores)
    df = pd.DataFrame({"WIDS": scores, "Group": labels})

    stat, pval = stats.mannwhitneyu(g_scores, h_scores, alternative="greater")

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Boxplot
    ax = axes[0]
    palette = {"Healthy": C["healthy"], "GDM": C["gdm"]}
    sns.boxplot(data=df, x="Group", y="WIDS", palette=palette,
                width=0.4, linewidth=1.5, ax=ax)
    sns.stripplot(data=df, x="Group", y="WIDS", color="black",
                  size=5, alpha=0.45, jitter=True, ax=ax)
    ax.set_title(f"Negative Control: Healthy vs GDM WIDS\n"
                 f"Mann-Whitney U: p = {pval:.2e}", fontsize=11,
                 fontweight="bold")
    ax.set_ylabel("WIDS score", fontsize=11)
    ax.set_facecolor(C["grid"])
    ax.grid(axis="y", color="white", lw=0.8)

    # Histogram
    ax = axes[1]
    ax.hist(h_scores, bins=15, alpha=0.65, color=C["healthy"],
            label=f"Healthy (n={len(h_scores)})", density=True)
    ax.hist(g_scores, bins=15, alpha=0.65, color=C["gdm"],
            label=f"GDM (n={len(g_scores)})", density=True)
    ax.set_xlabel("WIDS score", fontsize=11)
    ax.set_ylabel("Density", fontsize=11)
    ax.set_title("Score Distribution: Healthy vs GDM", fontsize=11,
                 fontweight="bold")
    ax.legend(fontsize=10)
    ax.set_facecolor(C["grid"])
    ax.grid(color="white", lw=0.8)

    fig.suptitle("WIDS Negative Control Validation", fontsize=13,
                 fontweight="bold")
    fig.tight_layout()
    _save(fig, path)


def fig_nirt_comparison(nirt_res, meta, outcomes, path):
    """
    Three-panel NIRT result figure.

    Panel 1: WIDS distributions for TB vs TBCOVID (boxplot + strip)
    Panel 2: WIDS bar chart for both groups ranked together
    Panel 3: WIDS vs adverse outcome (boxplot by outcome status)

    PURPOSE: Shows WIDS generalises from GDM (gut, 16S) to TB-COVID
    (sputum, WGS) without any modification to the algorithm.
    The hypothesis is that TBCOVID co-infection imposes an additional
    ecological disruption beyond TB alone, detectable as higher WIDS.
    """
    fig = plt.figure(figsize=(18, 6))
    gs  = fig.add_gridspec(1, 3, wspace=0.35)

    tb_s  = nirt_res['TB']['wids']
    tc_s  = nirt_res['TBCOVID']['wids']
    tb_p  = nirt_res['TB']['patients']
    tc_p  = nirt_res['TBCOVID']['patients']

    # Panel 1: Distribution comparison
    ax1 = fig.add_subplot(gs[0])
    df  = pd.DataFrame({
        "WIDS": np.concatenate([tb_s, tc_s]),
        "Group": ["TB"]*len(tb_s) + ["TBCOVID"]*len(tc_s)
    })
    palette = {"TB": C["tb"], "TBCOVID": C["tbcovid"]}
    sns.boxplot(data=df, x="Group", y="WIDS", palette=palette,
                width=0.4, linewidth=1.5, ax=ax1)
    sns.stripplot(data=df, x="Group", y="WIDS", color="black",
                  size=5, alpha=0.5, jitter=True, ax=ax1)
    stat, pval = stats.mannwhitneyu(tc_s, tb_s, alternative="greater")
    ax1.set_title(f"TB vs TBCOVID WIDS\nMann-Whitney p = {pval:.3f}",
                  fontsize=11, fontweight="bold")
    ax1.set_ylabel("WIDS score", fontsize=11)
    ax1.set_facecolor(C["grid"])
    ax1.grid(axis="y", color="white", lw=0.8)

    # Panel 2: Ranked bar chart
    ax2 = fig.add_subplot(gs[1])
    all_s = np.concatenate([tb_s, tc_s])
    all_p = tb_p + tc_p
    all_g = ["TB"]*len(tb_p) + ["TBCOVID"]*len(tc_p)
    order = np.argsort(all_s)[::-1]
    s_ord = all_s[order]
    c_ord = [C["tb"] if all_g[i]=="TB" else C["tbcovid"] for i in order]
    p_ord = [all_p[i] for i in order]
    ax2.bar(range(len(s_ord)), s_ord, color=c_ord, edgecolor="white", lw=0.3)
    ax2.set_xticks(range(len(p_ord)))
    ax2.set_xticklabels(p_ord, rotation=90, fontsize=5)
    ax2.set_ylabel("WIDS score", fontsize=10)
    ax2.set_title("WIDS Ranked — TB & TBCOVID", fontsize=11, fontweight="bold")
    ax2.set_facecolor(C["grid"])
    ax2.grid(axis="y", color="white", lw=0.8)
    patches_nirt = [mpatches.Patch(fc=C["tb"],      label="TB"),
                    mpatches.Patch(fc=C["tbcovid"],  label="TBCOVID")]
    ax2.legend(handles=patches_nirt, fontsize=8)

    # Panel 3: WIDS vs adverse outcome
    ax3 = fig.add_subplot(gs[2])
    rows = []
    for grp in ['TB', 'TBCOVID']:
        pats   = nirt_res[grp]['patients']
        wids   = nirt_res[grp]['wids']
        grp_out = outcomes[outcomes['Group'] == grp]['adverse'].reindex(pats)
        for p, w, o in zip(pats, wids, grp_out):
            if pd.notna(o):
                rows.append({'Group': grp, 'WIDS': w, 'Adverse': int(o),
                             'Outcome': 'Adverse' if o else 'No adverse'})
    if rows:
        df_out = pd.DataFrame(rows)
        sns.boxplot(data=df_out, x="Outcome", y="WIDS",
                    palette={"Adverse": C["outlier"], "No adverse": C["ctrl"]},
                    width=0.4, linewidth=1.5, ax=ax3)
        sns.stripplot(data=df_out, x="Outcome", y="WIDS", color="black",
                      size=5, alpha=0.5, jitter=True, ax=ax3)
        r_tb  = nirt_res['TB']['outcome_corr']
        r_tc  = nirt_res['TBCOVID']['outcome_corr']
        ax3.set_title(f"WIDS vs Adverse Outcome\nSpearman: TB r={r_tb:.2f}  "
                      f"TBCOVID r={r_tc:.2f}", fontsize=10, fontweight="bold")
        ax3.set_ylabel("WIDS score", fontsize=11)
        ax3.set_facecolor(C["grid"])
        ax3.grid(axis="y", color="white", lw=0.8)

    fig.suptitle("NIRT TB-COVID Sputum Microbiome — WIDS Generalisation Analysis",
                 fontsize=13, fontweight="bold")
    _save(fig, path)


def fig_summary_dashboard(aupr_best, hit_rate, jaccard_network,
                          mw_pval_ctrl, nirt_mw_pval, path):
    """
    One-slide summary dashboard for PhD presentation.

    Five metric tiles showing the key quantitative claims:
    1. AUPR of outlier detection
    2. Top-3 hit rate
    3. Network Jaccard (Pearson vs Spearman)
    4. Negative control p-value
    5. NIRT generalisation p-value
    """
    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    metrics = [
        (f"{aupr_best:.3f}", "AUPR\n(outlier recovery)", C["outlier"],
         "Analysis 1\nUnsupervised g2/g13/g23\nrecovery"),
        (f"{hit_rate}/3",    "Ground-truth\nhit count",   C["gdm"],
         "Analysis 1\nTop-3 WIDS vs\nFig 6 outliers"),
        (f"{jaccard_network:.3f}", "Jaccard\nPearson↔Spearman", C["alpha"],
         "Analysis 2\nNetwork structure\ndifference"),
        (f"p={mw_pval_ctrl:.2e}", "Healthy ctrl\np-value",  C["healthy"],
         "Analysis 5\nGDM > Healthy\nMann-Whitney U"),
        (f"p={nirt_mw_pval:.3f}", "NIRT TBCOVID\nvs TB p-value", C["tbcovid"],
         "Analysis 6\nGeneralisation to\nsputum microbiome"),
    ]
    for ax, (val, label, col, note) in zip(axes, metrics):
        ax.set_facecolor(col)
        ax.text(0.5, 0.58, val, ha="center", va="center",
                fontsize=24, fontweight="bold", color="white",
                transform=ax.transAxes)
        ax.text(0.5, 0.30, label, ha="center", va="center",
                fontsize=11, color="white", transform=ax.transAxes,
                multialignment="center")
        ax.text(0.5, 0.08, note, ha="center", va="center",
                fontsize=7.5, color="white", alpha=0.85,
                transform=ax.transAxes, multialignment="center")
        ax.set_xticks([]); ax.set_yticks([])
        for sp in ax.spines.values():
            sp.set_edgecolor("white"); sp.set_linewidth(2)

    fig.suptitle("WIDS Pipeline — PhD Presentation Summary",
                 fontsize=14, fontweight="bold", y=1.02)
    fig.tight_layout()
    _save(fig, path)


In [25]:
# ══════════════════════════════════════════════════════════════════════════════
# 8.  HELPER
# ══════════════════════════════════════════════════════════════════════════════

def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


def save_csv(df_or_dict, path):
    os.makedirs(os.path.dirname(path) if os.path.dirname(path) else ".", exist_ok=True)
    if isinstance(df_or_dict, dict):
        pd.Series(df_or_dict).to_csv(path, header=["value"])
    else:
        df_or_dict.to_csv(path, index=False)
    log(f"  CSV → {path}")


In [26]:
# ══════════════════════════════════════════════════════════════════════════════
# 9.  MAIN PIPELINE
# ══════════════════════════════════════════════════════════════════════════════

def main(args):
    out   = args.out
    figs  = os.path.join(out, "figures")
    csvs  = os.path.join(out, "tables")
    for d in [figs, csvs]: os.makedirs(d, exist_ok=True)
    N = args.shuffle

    # ──────────────────────────────────────────────────────────────────────
    # GDM DATA LOAD
    # ──────────────────────────────────────────────────────────────────────
    log("═"*60)
    log("LOADING GDM DATA")
    log("═"*60)
    otu, meta, rich = load_gdm(args.otu_gdm, args.meta_gdm, args.rich_meta)

    gdm_W0  = subset_mat(otu, meta, "disease", "GDM",     week_val="W0")
    gdm_W2  = subset_mat(otu, meta, "disease", "GDM",     week_val="W2")
    hlt_W0  = subset_mat(otu, meta, "disease", "healthy", week_val="W0")
    hlt_W2  = subset_mat(otu, meta, "disease", "healthy", week_val="W2")

    log(f"GDM W0:{gdm_W0.shape} | GDM W2:{gdm_W2.shape} | "
        f"Hlt W0:{hlt_W0.shape} | Hlt W2:{hlt_W2.shape}")
    # ──────────────────────────────────────────────────────────────────────
    # ANALYSIS 2: SPEARMAN vs PEARSON
    # ──────────────────────────────────────────────────────────────────────
    log("\n" + "═"*60)
    log("ANALYSIS 2: Spearman vs Pearson Network Comparison")
    log("═"*60)
    net_stats, adj_p, adj_s, deg_p, deg_s = compare_pearson_spearman(
        gdm_W0, n_shuffle=N)
    log(f"  Jaccard overlap : {net_stats['jaccard_overlap']:.4f}")
    log(f"  Degree KS p     : {net_stats['degree_ks_pval']:.4f}")
    save_csv(net_stats, os.path.join(csvs, "network_comparison.csv"))
    fig_network_comparison(deg_p, deg_s, net_stats,
                           list(gdm_W0.index),
                           os.path.join(figs, "A2_network_comparison.png"))
 # ──────────────────────────────────────────────────────────────────────
    # ANALYSIS 1 + 4: WIDS at W0 (Pearson baseline, then Spearman)
    # ──────────────────────────────────────────────────────────────────────
    log("\n" + "═"*60)
    log("ANALYSIS 1+4: WIDS at W0 — Outlier Detection + Alpha Grid")
    log("═"*60)

    for method in ["pearson", "spearman"]:
        log(f"\n  -- method = {method} --")
        jd, ji, wgrid, pats_W0, fn, rn = compute_wids(
            gdm_W0, hlt_W0, method=method,
            alpha_vals=ALPHA_GRID, n_shuffle=N, label=f"GDM-W0-{method}")

        aupr_vals = np.array([compute_aupr(wgrid[:, i], pats_W0, GT_W0)
                               for i in range(len(ALPHA_GRID))])
        best_i    = int(np.nanargmax(aupr_vals))
        best_a    = ALPHA_GRID[best_i]
        best_aupr = aupr_vals[best_i]
        best_sc   = wgrid[:, best_i]

        log(f"  Best α={best_a:.2f}  AUPR={best_aupr:.4f}")

        # Permutation p-value
        obs_aupr, perm_p = permutation_pvalue(best_sc, pats_W0, GT_W0,
                                               n_perm=1000)
        log(f"  Permutation p-value = {perm_p:.4f}")

        df_w0 = pd.DataFrame({
            "patient": pats_W0,
            "j_direct":   jd,
            "j_indirect": ji,
            "wids":       best_sc,
            "ground_truth": [int(p in GT_W0) for p in pats_W0],
        }).sort_values("wids", ascending=False).reset_index(drop=True)
        save_csv(df_w0, os.path.join(csvs, f"wids_W0_{method}.csv"))

        df_alpha = pd.DataFrame({"alpha": ALPHA_GRID, "aupr": aupr_vals})
        save_csv(df_alpha, os.path.join(csvs, f"alpha_sensitivity_{method}.csv"))

        fig_wids_bar(best_sc, pats_W0,
                     f"WIDS Scores — GDM W0 ({method.title()}, α={best_a:.2f}  "
                     f"AUPR={best_aupr:.3f}  perm-p={perm_p:.3f})",
                     os.path.join(figs, f"A1_wids_W0_{method}.png"),
                     ground_truth=GT_W0)
        fig_alpha_sensitivity(ALPHA_GRID, aupr_vals, best_a,
                              os.path.join(figs, f"A4_alpha_{method}.png"))

        # Store Spearman results for later analyses
        if method == "spearman":
            best_sc_W0_spearman = best_sc
            pats_W0_sp          = pats_W0
            best_a_sp           = best_a
            best_aupr_sp        = best_aupr
            perm_p_sp           = perm_p
            jd_W0_sp            = jd
            ji_W0_sp            = ji

    # ──────────────────────────────────────────────────────────────────────
    # ANALYSIS 3: DELTA WIDS (W0 → W2)
    # ──────────────────────────────────────────────────────────────────────
    log("\n" + "═"*60)
    log("ANALYSIS 3: Diet Intervention Effect — ΔWIDS (W0 → W2)")
    log("═"*60)
    jd_W2, ji_W2, wgrid_W2, pats_W2, fn_W2, rn_W2 = compute_wids(
        gdm_W2, hlt_W2, method="spearman",
        alpha_vals=ALPHA_GRID, n_shuffle=N, label="GDM-W2-spearman")

    best_idx = np.argmin(np.abs(ALPHA_GRID - best_a_sp))
    sc_W2    = wgrid_W2[:, best_idx]

    # Match patients W0 ↔ W2 by base ID (G1.1 → G1)
    def base(p): return p.rsplit(".", 1)[0]
    w0_map = {base(p): s for p, s in zip(pats_W0_sp, best_sc_W0_spearman)}
    w2_map = {base(p): s for p, s in zip(pats_W2,    sc_W2)}
    matched = sorted(set(w0_map) & set(w2_map))
    delta   = np.array([w2_map[b] - w0_map[b] for b in matched])

    df_delta = pd.DataFrame({
        "patient": matched,
        "wids_W0": [w0_map[b] for b in matched],
        "wids_W2": [w2_map[b] for b in matched],
        "delta_wids": delta,
    }).sort_values("delta_wids", ascending=False).reset_index(drop=True)
    save_csv(df_delta, os.path.join(csvs, "delta_wids.csv"))
    save_csv(pd.DataFrame({"patient": pats_W2, "wids": sc_W2}),
             os.path.join(csvs, "wids_W2_spearman.csv"))

    fig_delta_wids(delta, matched,
                   os.path.join(figs, "A3_delta_wids.png"))
    fig_wids_bar(sc_W2, pats_W2,
                 f"WIDS Scores — GDM W2 (Spearman, α={best_a_sp:.2f})",
                 os.path.join(figs, "A3_wids_W2_spearman.png"),
                 ground_truth=GT_W2)

    log(f"  Matched {len(matched)} patients W0↔W2")
    log(f"  Top ΔWIDS: {df_delta[['patient','delta_wids']].head(5).to_string()}")

    # ──────────────────────────────────────────────────────────────────────
    # ANALYSIS 5: HEALTHY NEGATIVE CONTROL
    # ──────────────────────────────────────────────────────────────────────
    log("\n" + "═"*60)
    log("ANALYSIS 5: Healthy Negative Control")
    log("═"*60)
    jd_h, ji_h, wgrid_h, pats_h, fn_h, rn_h = compute_wids(
        hlt_W0, hlt_W0, method="spearman",
        alpha_vals=ALPHA_GRID, n_shuffle=N, label="Healthy-ctrl")

    h_scores = wgrid_h[:, best_idx]
    stat_mw, pval_mw = stats.mannwhitneyu(
        best_sc_W0_spearman, h_scores, alternative="greater")

    log(f"  Healthy WIDS  mean={h_scores.mean():.4f}  std={h_scores.std():.4f}")
    log(f"  GDM    WIDS  mean={best_sc_W0_spearman.mean():.4f}")
    log(f"  Mann-Whitney U : stat={stat_mw:.1f}  p={pval_mw:.2e}")

    save_csv(pd.DataFrame({"patient": pats_h, "wids": h_scores}),
             os.path.join(csvs, "healthy_wids.csv"))
    fig_negative_control(h_scores, pats_h,
                         best_sc_W0_spearman, pats_W0_sp,
                         os.path.join(figs, "A5_negative_control.png"))

    # ──────────────────────────────────────────────────────────────────────
    # ANALYSIS 6 + 7: NIRT TB-COVID
    # ──────────────────────────────────────────────────────────────────────
    log("\n" + "═"*60)
    log("ANALYSIS 6+7: NIRT TB-COVID Sputum Microbiome — WIDS Generalisation")
    log("═"*60)
    genus_clr, nirt_meta, outcomes = load_nirt(
        args.meta_nirt, args.genus, args.outcomes)

    nirt_res, ctrl_mat = run_nirt_wids(
        genus_clr, nirt_meta, outcomes,
        n_shuffle=N, alpha_vals=ALPHA_GRID, best_alpha=best_a_sp)

    # Save tables
    for grp, res in nirt_res.items():
        df_nirt = pd.DataFrame({
            "patient": res['patients'],
            "j_direct":   res['j_direct'],
            "j_indirect": res['j_indirect'],
            "wids":       res['wids'],
        }).sort_values("wids", ascending=False).reset_index(drop=True)
        save_csv(df_nirt, os.path.join(csvs, f"nirt_wids_{grp}.csv"))

    _, nirt_mw_pval = stats.mannwhitneyu(
        nirt_res['TBCOVID']['wids'], nirt_res['TB']['wids'],
        alternative="greater")
    log(f"  TBCOVID vs TB Mann-Whitney p = {nirt_mw_pval:.4f}")

    fig_nirt_comparison(nirt_res, nirt_meta, outcomes,
                        os.path.join(figs, "A6_nirt_wids.png"))
    # ──────────────────────────────────────────────────────────────────────
    # SUMMARY DASHBOARD
    # ──────────────────────────────────────────────────────────────────────
    top3_set, hit = fig_wids_bar.__wrapped__ if hasattr(fig_wids_bar,'__wrapped__') else (set(), set())
    # Recompute top3/hit for dashboard
    order = np.argsort(best_sc_W0_spearman)[::-1]
    top3_dash = {pats_W0_sp[i] for i in order[:3]}
    hit_dash  = top3_dash & GT_W0

    fig_summary_dashboard(
        aupr_best    = best_aupr_sp,
        hit_rate     = len(hit_dash),
        jaccard_network = net_stats["jaccard_overlap"],
        mw_pval_ctrl = pval_mw,
        nirt_mw_pval = nirt_mw_pval,
        path         = os.path.join(figs, "Z_summary_dashboard.png")
    )
    # ──────────────────────────────────────────────────────────────────────
    # FINAL SUMMARY
    # ──────────────────────────────────────────────────────────────────────
    log("\n" + "═"*60)
    log("PIPELINE COMPLETE — FINAL SUMMARY")
    log("═"*60)
    log(f"  [A1] Best α (Spearman)         : {best_a_sp:.2f}")
    log(f"  [A1] Best AUPR                 : {best_aupr_sp:.4f}")
    log(f"  [A1] Permutation p-value       : {perm_p_sp:.4f}")
    log(f"  [A1] Top-3 patients            : {top3_dash}")
    log(f"  [A1] Ground-truth outliers     : {GT_W0}")
    log(f"  [A1] Hit rate                  : {len(hit_dash)}/3")
    log(f"  [A2] Jaccard Pearson↔Spearman  : {net_stats['jaccard_overlap']:.4f}")
    log(f"  [A2] Degree KS p-value         : {net_stats['degree_ks_pval']:.4f}")
    log(f"  [A3] Max |ΔWIDS|               : {np.abs(delta).max():.4f}")
    log(f"  [A5] GDM vs Healthy p-value    : {pval_mw:.2e}")
    log(f"  [A6] TBCOVID vs TB p-value     : {nirt_mw_pval:.4f}")
    log(f"  [A7] TB outcome Spearman r     : {nirt_res['TB']['outcome_corr']:.3f}")
    log(f"  [A7] TBCOVID outcome Spearman r: {nirt_res['TBCOVID']['outcome_corr']:.3f}")
    log(f"\n  Results in: {os.path.abspath(out)}/")



In [27]:
# ══════════════════════════════════════════════════════════════════════════════
# 10. CLI
# ══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    # In a notebook environment, parse_args() will fail if no arguments are passed.
    # We'll create a dummy args object with default values and correct paths.
    class Args:
        def __init__(self):
            # Corrected paths: files are in the root /content/ directory
            self.otu_gdm = 'otu_table.txt'
            self.meta_gdm = 'group_type.txt'
            self.rich_meta = 'liu2023_metadata.csv' # Assuming this file is also in /content/
            self.meta_nirt = 'SupplFileD1b.xlsx'
            self.genus = 'SupplFileD3.xlsx'
            self.outcomes = 'SupplFileD7.xlsx'
            self.out = 'results/'
            self.shuffle = N_SHUFFLE

    args = Args()

    # N_SHUFFLE is a global constant, needs to be updated with the value from args
    N_SHUFFLE = args.shuffle
    main(args)

[03:45:36] ════════════════════════════════════════════════════════════
[03:45:36] LOADING GDM DATA
[03:45:36] ════════════════════════════════════════════════════════════
[03:45:36] GDM OTU table : 101 OTUs × 114 samples
[03:45:36] GDM groups    : {'h(W0)': 30, 'h(W2)': 30, 'g(W0)': 27, 'g(W2)': 27}
[03:45:36] GDM W0:(101, 27) | GDM W2:(101, 27) | Hlt W0:(101, 30) | Hlt W2:(101, 30)
[03:45:36] 
════════════════════════════════════════════════════════════
[03:45:36] ANALYSIS 2: Spearman vs Pearson Network Comparison
[03:45:36] ════════════════════════════════════════════════════════════
[03:45:36]   Building Pearson network for GDM W0 …
[03:45:37]   Building Spearman network for GDM W0 …
[03:45:37]   Jaccard overlap : 0.3665
[03:45:37]   Degree KS p     : 0.2159
[03:45:37]   CSV → results/tables/network_comparison.csv
[03:45:39]   Saved → results/figures/A2_network_comparison.png
[03:45:39] 
════════════════════════════════════════════════════════════
[03:45:39] ANALYSIS 1+4: WIDS at W